# 18 - Reading the first night on sky

**Purpose.** To explain what notebook `17` established from `data/session06`, and what a reader
should now believe about the three sky terms in MISSION's model. `17` is the notebook that read
the frames and made these numbers, and is written for someone *checking* the work. This one is
written for someone *deciding what to do next* - which, after this session, is a question with a
sharp answer.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `sky_constants.json`, `sky_frames.csv`, `sky_pairs.csv`, and the predecessors they
lean on: `cold_constants.json` for `g` and the pedestal at this night's setpoint,
`dark_constants.json` for the dark bound and the stacking ladder. If any of it disagreed with
`results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`** for why a modal level over 65,000 pixels resolves a fraction
of a count, and `16_cold_constants_read.ipynb` for why the constants come from a -20 C bench run
rather than from session 02.

**The shape of this session, stated before any number.** It delivered two of MISSION's three sky
constants outright, one of them only as a prediction, and **none of the four ranked pairs** - which
is the definition of done. That is not a failure of the night. The frames are there, the cells are
balanced, the ladder reaches. What is missing is an *engine*: registering and integrating frames
is build step 5, `astropix/pixinsight.py` and `pjsr/`, and it does not exist. Section 6 is about
exactly that, because it is the whole of what stands between this repo and its own finish line.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
read = lambda n: json.loads((RESULTS / n).read_text())

K6 = read("sky_constants.json")            # notebook 17, this session
K7 = read("cold_constants.json")           # notebook 15, the constants it consumed
K3 = read("dark_constants.json")           # session 03, the dark bound and the stacking ladder

frames = pd.read_csv(RESULTS / "sky_frames.csv")
pairs = pd.read_csv(RESULTS / "sky_pairs.csv")

PLANES = ["R", "G1", "G2", "B"]
CELLS = ["A", "B", "C", "D"]
CELL_SETTING = {"A": (50, 30), "B": (50, 120), "C": (200, 30), "D": (200, 120)}
L32 = {"R": 1.500, "green": 1.594, "B": 0.910}

F_SKY = K6["F_sky"]["value"]
T_DEAD = K6["t_dead"]["value"]
good = frames[~frames.rejected].copy()

print("notebook 17 published %d constants on %s, from %d frames"
      % (len(K6), K6["t_dead"]["measured_on"], K6["t_dead"]["source_frames"]))
nulls = [k for k, v in K6.items() if v["value"] is None]
print("  %d measured, %d published null with a reason" % (len(K6) - len(nulls), len(nulls)))
print("  nulls: " + ", ".join(nulls))
print()
print("constants it consumed, and where each came from:")
print(f"  g, pedestal, R   cold_constants.json at {K7['setpoint']['value']} C "
      f"(notebook 15) -- NOT session 02")
print(f"  dark current     {K3['dark_current_bound']['value']} e-/px/s at -10 C, "
      "an upper bound and colder here still")

## 1. Four ways the night was not the protocol, and what each cost

None of these was discovered by a gate. That is the uncomfortable part and it is worth sitting
with: every one of them is invisible to a check that looks at one frame at a time.

In [ ]:
dev = K6["setpoint_deviation"]["value"]
roi = K6["sky_roi"]["value"]

print("1. SETPOINT")
print(f"   protocol {dev['protocol_c']} C, actual {dev['actual_c']} C, on every frame.")
print(f"   frames outside their OWN band: {dev['frames_outside_band']} -- the cooler was "
      "perfect.")
print("   cost: a whole bench session (07) to measure the constants where the frames were.")

print("\n2. BIAS BRACKETS")
print("   the protocol asks for 20 at each gain at each end of the night.  None were shot.")
print("   cost: the pedestal comes from a bench block on another night at the same setpoint,")
print(f"   carrying about a count of uncertainty -- which is "
      f"{100 * K6['F_sky']['uncertainty']['G1'] / F_SKY['G1']:.1f}% of F_sky on G1.")

print("\n3. MERIDIAN FLIP")
print(f"   {K6['frames_dropped_to_flip']['value']} frames dropped: the field rotated and gate 4")
print("   fixes both ROIs for the night.  cost: the smallest of the four, and every cell still")
print("   clears the floor of 24.")

print("\n4. GUIDING")
print("   the ASIAIR writes no guide RMS into the header, so gate 3's guiding half could not")
print("   run at all.  cost: frames are rejected on the sky level (cloud) and nothing else.")
print(f"   rejected: {json.dumps(K6['rejection']['value'])}")

## 2. `t_dead` is the headline, and it is worse than the archive predicted

MISSION spends the sub-exposure question in two opposing terms. `R^2/t` says lengthen the sub -
read noise is paid once per sub, so fewer subs means less of it. `t_dead` says the opposite: at
fixed wall clock, every sub costs its overhead whether or not it collects anything.

```
SNR(T_night, t) ∝ sqrt(t / (t + t_dead)) / sqrt(F_obj + F_sky + D + R^2/t)
```

The archive - 2,002 frames over seven nights - measured 17 to 20 s, dithering every second frame.
**This night dithered after every frame**, deliberately: the protocol's reasoning is that a dither
on some frames and not others means whichever cell sits after a dither pays an overhead the others
do not, and that overhead lands inside `t_dead` where it cannot be separated from the setting.
Dithering uniformly makes `t_dead` one number and makes it fair.

The price of that fairness is below, and it is the number that should change how the rig is run.

In [ ]:
dec = K6["t_dead_decomposition"]["value"]
print(f"t_dead                      {T_DEAD:7.2f} s   +/- {K6['t_dead']['uncertainty']:.2f}")
print(f"  shortest gap all night    {dec['download_and_save_bound']:7.2f} s   "
      "(upper bound on download)")
print(f"  archive bare download     {dec['download_and_save_archive']:7.2f} s   "
      "(independent, not ours)")
print(f"  dither settle, inferred   {dec['dither_settle_inferred']:7.2f} s")
print(f"including interruptions     {K6['t_dead_including_interruptions']['value']:7.2f} s")
print(f"\narchive, dithering every second frame: 17-20 s")

print("\nwhat it costs, at the two sub lengths this night shot:")
for t in (30, 120):
    lost = T_DEAD / (t + T_DEAD)
    print(f"  {t:3d} s subs: {100 * lost:4.1f}% of the night collects nothing, "
          f"and SNR is scaled by sqrt(t/(t+t_dead)) = {np.sqrt(1 - lost):.3f}")

fig, ax = plt.subplots(figsize=(5.2, 3.0))
t = np.linspace(5, 600, 400)
for td, label in ((0.68, "0.68 s, bare download"), (19.0, "19 s, archive"),
                  (T_DEAD, f"{T_DEAD:.1f} s, this night")):
    ax.plot(t, np.sqrt(t / (t + td)), lw=1.2, label=label)
ax.set(xscale="log", xlabel="sub length, s", ylabel="duty factor  sqrt(t/(t+t_dead))",
       title="what dead time does to the sub-length choice")
ax.axvline(30, color="0.7", lw=0.8)
ax.axvline(120, color="0.7", lw=0.8)
ax.legend(fontsize=7)
fig.tight_layout()

print("\nthe curve is the whole argument for long subs on this rig, and it is not about read")
print("noise at all.  At 30 s, half the night is overhead.  At 120 s, a fifth.")

### Is `t_dead` one number, or four?

The interleave is only fair if every cell pays the same overhead. That was the reason for
dithering uniformly, so it is worth checking that it worked rather than assuming it.

In [ ]:
per_cell = K6["t_dead_per_cell"]["value"]
spread = K6["t_dead_per_cell"]["uncertainty"]
tbl = pd.DataFrame({"t_dead_s": per_cell})
tbl["setting"] = [f"gain {CELL_SETTING[c][0]}, {CELL_SETTING[c][1]} s" for c in tbl.index]
tbl["vs_overall_pct"] = 100 * (tbl.t_dead_s / T_DEAD - 1)
print(tbl.round(2).to_string())
print(f"\nspread across cells {spread:.2f} s, {100 * spread / T_DEAD:.0f}% of t_dead")
print("A ratio between two cells inherits the difference between their own overheads, so the")
print("pair predictions in section 6 are computed with the single t_dead and this table is the")
print("size of what that approximation hides.")

## 3. `F_sky`, and what happened to L32

The sky rate, per CFA plane, measured from the frames themselves rather than imported. It is an
**upper bound** on true sky and will always be one on this target: unresolved nebulosity sits
inside the sky box and cannot be separated. That is the right bound for the model, which needs the
level under the faint signal rather than the zodiacal sky in the abstract.

`LEGACY.md`'s **L32** is the one inherited claim this session consumes: 1.594 e-/px/s green, 1.500
red, 0.910 blue, under the same suburban sky at the same focal ratio and pixel scale. It came from
a codebase with no provenance discipline, and its origin tree is deleted - so this is the only
check it will ever get.

Two things are being tested, and only one of them is the number. L32 was explicit that its figure
was "a rate for that night, at that altitude", and that it fell about 5% across two hours as the
target rose. **Reproducing that shape is as much the test as reproducing the value.**

In [ ]:
cmp = K6["L32_comparison"]["value"]
tab = pd.DataFrame(cmp).T
print(tab.round(4).to_string())
print()
print(f"F_sky green over the night: {K6['F_sky_green']['value']:.4f} e-/px/s "
      f"(frame-to-frame sd {K6['F_sky_green']['uncertainty']:.4f})")
print(f"trend: {K6['F_sky_trend']['value']:+.4f} e-/px/s per hour")
print()
print("per cell -- four independent measurements of one sky, at two gains and two sub lengths:")
per = pd.DataFrame(K6["F_sky_per_cell"]["value"]).T
per["green"] = per[["G1", "G2"]].mean(axis=1)
print(per.round(4).to_string())
agree = 100 * (per.green.max() / per.green.min() - 1)
print(f"\nthe four agree to {agree:.1f}%.")
print("  " + ("That is the estimator validating itself: the same sky measured through two "
              "gains\n  that differ by a factor of six in g, and two sub lengths that differ by "
              "four, has\n  no business agreeing this well unless the pedestal, the gain and "
              "the mode are all right."
              if agree < 10 else
              "That is a wide spread for one sky.  Something gain- or exposure-dependent is in\n"
              "  the estimator -- suspect the pedestal first, since it is the term that does not\n"
              "  scale with either."))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.1))
for cell, d in good.groupby("cell"):
    ax[0].plot(d.minutes / 60, d.F_sky_green, ".", ms=3,
               label=f"{cell}: g{CELL_SETTING[cell][0]} {CELL_SETTING[cell][1]}s")
ax[0].axhline(L32["green"], color="k", lw=0.8, ls="--", label="L32, 1.594")
ax[0].set(xlabel="hours into the night", ylabel="F_sky green, e-/px/s",
          title="the sky rate across the night")
ax[0].legend(fontsize=6)

night_mean = {p: F_SKY[p] for p in PLANES}
night_mean["green"] = K6["F_sky_green"]["value"]
x = np.arange(len(L32))
ax[1].bar(x - 0.2, [night_mean[k] for k in L32], 0.4, label="measured")
ax[1].bar(x + 0.2, [L32[k] for k in L32], 0.4, label="L32")
ax[1].set_xticks(x)
ax[1].set_xticklabels(list(L32))
ax[1].set(ylabel="e-/px/s", title="against the inherited rate")
ax[1].legend(fontsize=7)
fig.tight_layout()

print("Sky brightness is weather, moon and town lighting.  A disagreement of tens of percent is")
print("a different night, not a wrong measurement -- which is exactly why L32 was never going")
print("to be a constant, and why MISSION lists F_sky as extracted per frame from the lights")
print("themselves.  What this session settles is that the estimator works and the magnitude is")
print("right.  What it cannot settle is a number for 'the suburban sky', because there isn't one.")

## 4. MISSION's third assumption survives, and it is the one that shapes the model

> **The dimmest and the brightest plane are different planes.** True of this sensor under these
> skies, or the Pareto gap between floor and ceiling closes and the per-plane framing buys
> nothing.

The assumption matters because the two constraints bind on different planes. The exposure *floor*
- how long a sub must be before sky shot noise swamps read noise - is set by the **dimmest**
plane, because that is the one that takes longest to get there. The clipping *ceiling* is set by
the **brightest**, because that is the one whose star cores fill first. If the planes agreed, both
constraints would land on the same channel, the gap between them would close, and a rule written
for a mono camera would lose nothing.

In [ ]:
print(f"planes differ by {K6['planes_differ']['value']:.1f}%")
d = pd.Series({p: F_SKY[p] for p in PLANES}).sort_values()
print(d.round(4).to_string())
print(f"\ndimmest {d.index[0]} sets the exposure floor; brightest {d.index[-1]} sets the ceiling")

R_e = {int(k): v for k, v in K7["read_noise_cold_e"]["value"].items()}
print("\nwhat that means for the floor, per plane -- the sub length at which sky shot noise")
print("reaches the read noise (m = F_sky*t/R^2 = 1):")
floor = pd.DataFrame({
    g: {p: R_e[g] ** 2 / F_SKY[p] for p in PLANES} for g in sorted(R_e)})
floor.columns = [f"gain {g}" for g in floor.columns]
print(floor.round(2).to_string())
print("\nseconds.  The spread down each column is the whole reason the per-plane framing exists.")

## 5. The star-colour half of the Pareto point

MISSION's criterion is SNR **subject to a star-colour constraint**: no more than a chosen fraction
of stars clipping their cores, since a clipped core renders white and no processing recovers it.
Gain 200 has about a sixth of the well of gain 50, so it must clip far more of the same field at
the same sub length. This is the first time this project puts a number on that.

**It is the pixel-level number, not the star-level one, and the difference is not cosmetic.** The
protocol asks for the fraction of *stars* with a clipped core against the same detection list;
the same physical star sits on a different pixel in every frame because the night dithered after
each one, so matching a list across frames is registration - build step 5 again. The pixel count
understates the star count in a knowable direction: a clipped core occupies a handful of pixels,
so a clipped fraction of 0.1% is a great many clipped stars.

In [ ]:
clip = pd.DataFrame(K6["clipped_fraction"]["value"]).T[PLANES]
clip["setting"] = [f"gain {CELL_SETTING[c][0]}, {CELL_SETTING[c][1]} s" for c in clip.index]
print("fraction of pixels at the top code, signal ROI:")
print(clip.round(6).to_string())

print("\nthe trade, at matched sub length:")
for t, lo, hi in ((30, "A", "C"), (120, "B", "D")):
    a = float(np.mean([K6["clipped_fraction"]["value"][lo][p] for p in PLANES]))
    c = float(np.mean([K6["clipped_fraction"]["value"][hi][p] for p in PLANES]))
    print(f"  {t:3d} s: gain 50 {a:.5f}  ->  gain 200 {c:.5f}   ({c / a:.1f}x more clipping)")

print("\nand the brightest plane is the one that gets there first, which is MISSION's point:")
worst = clip[PLANES].idxmax(axis=1)
print(pd.DataFrame({"worst plane": worst,
                    "its clipped fraction": clip[PLANES].max(axis=1)}).round(6).to_string())

## 6. Which pairs are still tests - and why none of them is a test yet

MISSION's definition of done: **the model must predict the SNR ranking of two settings to within
10%, on at least three pairs, one straddling the HCG threshold**, and each pair must be one the
model predicts *apart* - the predicted separation must exceed the measured repeatability of the
SNR estimator.

This session was designed to deliver four pairs, two of them straddling. It delivers **none**, and
the reason is a missing engine rather than a missing night. But it does something the protocol
could not: it replaces the predicted separations, which were computed from L32's sky rate and a
bracketed `t_dead` of 0.7 to 19 s, with predictions from **measured** constants.

That changes the answer to "which of these is even a test". The protocol flagged C -> D as a
near-tie at 1.6% if dead time were small. Dead time is not small.

In [ ]:
p = pairs.pivot_table(index=["pair", "why"], columns="plane", values="predicted_pct")
print("predicted separation, %, from this session's own F_sky and t_dead:")
print(p.round(2).to_string())

print(f"\nall computed at the measured t_dead of {T_DEAD:.1f} s.")
print("The protocol's own table, from L32 and a bracketed t_dead, predicted:")
print("  A->B  +16.7% (0.7 s) to +37.4% (19 s)   |  C->D  +1.6% to +19.7%")
print("  A->C  +21.6% at either                  |  B->D  +5.9% at either")

print("\nWHAT IS STILL MISSING, and it is the same thing three times:")
for k in ("eta_comb_registered", "snr_repeatability", "ranked_pairs"):
    print(f"  {k}: {K6[k]['value']}")
print("\nUntil snr_repeatability exists, none of the numbers above can be called a test even")
print("when the measured column is filled: MISSION requires the separation to beat the")
print("estimator's own repeatability, and that bar is currently unknown.  A pair the model")
print("calls a tie is a null result every model passes.")

### The gap that was this session's actual deliverable

Session 03 published `eta_comb` on darks - no registration, no resampling, so an **upper bound by
construction**. The number this session was supposed to add is the same measurement on registered,
dithered lights. The gap between the two *is* the resampling loss, and there is no other way to
get it.

In [ ]:
ladder = {int(k): v for k, v in K3["eta_comb"]["value"].items()}
N = [2, 4, 8, 16, 32]
tab = pd.DataFrame({
    "darks (session 03)": {n: ladder.get(n) for n in N},
    "ideal sqrt(N)": {n: 1.0 for n in N},
    "registered lights": {n: np.nan for n in N},
    "resampling loss": {n: np.nan for n in N},
})
print(tab.to_string())
smallest = min(K6["rejection"]["value"][c]["frames"] - K6["rejection"]["value"][c]["rejected"]
               for c in K6["rejection"]["value"])
print(f"\nthe ladder needs {max(N)} frames per cell and the thinnest cell has {smallest}.")
print("The data is sufficient.  Only the engine is missing.")

## 7. What build step 5 has to deliver, and what it already knows

This is the bottleneck, so it is worth stating exactly what is needed rather than "PixInsight
integration". `LEGACY.md` holds nine entries for it - L16 to L24 - harvested from the retired
attempts and not one of them verified here. They are the cheapest part of the work, because they
are all failures somebody has already paid for.

| needed | what it unblocks |
|---|---|
| a PJSR harness that launches, times out and reports by file | everything below |
| registration of the four cells onto a common frame | `eta_comb`, and any stacked SNR |
| integration with recorded rejection settings | `eta_comb`'s provenance, which is not independent of them |
| two half-stacks per cell | `snr_repeatability`, and therefore the bar every pair is judged against |
| a star detection list matched across frames | the star-level clipping fraction section 5 could only bound |

In [ ]:
print("what the retired attempts already learned, waiting in LEGACY.md:")
for k, v in {
    "L16": "the CLI invocation, and the flag that hangs a headless run forever",
    "L17": "never wait on PixInsight unbounded, and never trust its exit code",
    "L18": "a PJSR script must report by writing a file -- on both paths",
    "L19": "parameters go in a JSON file, and numeric types survive",
    "L20": "scale by 65535, not 65520 -- and PI does not debayer on load",
    "L21": "SplitCFA returns R, G2, G1, B, and medians will not catch the error",
    "L22": "PixInsight's variance divides by n-1; numpy's divides by n",
    "L23": "subtracting two 16-bit images clips every negative difference",
    "L24": "PI's own noise estimators, and how it chooses between them",
}.items():
    print(f"  {k}  {v}")
print("\nNone of these is verified here.  Each is a prediction to falsify, and together they")
print("are most of a session's worth of debugging that has already been paid for once.")

## 8. What is settled, and what the next session inherits

In [ ]:
print("SETTLED, WITH PROVENANCE")
print(f"  F_sky per plane, e-/px/s: "
      + ", ".join(f"{p} {F_SKY[p]:.3f}" for p in PLANES))
print(f"  F_sky green:              {K6['F_sky_green']['value']:.3f} "
      f"(L32 said {L32['green']}, "
      f"{K6['L32_comparison']['value']['green']['pct']:+.0f}%)")
print(f"  t_dead:                   {T_DEAD:.2f} s, mean, this dither cadence")
print(f"  planes differ by:         {K6['planes_differ']['value']:.0f}%  "
      "-> MISSION's third assumption holds")
print(f"  clipping, gain 200 vs 50: see section 5")
print()
print("NOT SETTLED, AND THE REASON IS ONE MISSING COMPONENT")
print("  eta_comb on registered lights, the SNR estimator's repeatability, and all four ranked")
print("  pairs.  Build step 5: astropix/pixinsight.py and pjsr/.")
print()
print("NOT SETTLED, AND THE REASON IS THE NIGHT")
print("  guide RMS per frame, so gate 3's guiding rejection was never applied.  Next sky")
print("  session should export the ASIAIR guiding log alongside the frames, or the gate is")
print("  unrunnable again.")
print("  bias brackets, so the pedestal is a bench number on another night.  They cost under a")
print("  minute and they are the term F_sky is most exposed to.")
print(f"  a sky ROI on genuinely dark sky.  Darkest to brightest corner is only "
      f"{roi['corner_range_counts']:.1f} counts")
print(f"  out of {good.sky_mode_green.mean():.0f}, and the two darkest are separated by "
      f"{roi['margin_over_second_counts']:.2f} against a")
print(f"  standard error of {roi['margin_standard_error_counts']:.2f} -- so no corner is "
      "demonstrably free of nebulosity and")
print("  F_sky carries more of it than the protocol intended.")
print()
print("WHAT THE NEXT SKY SESSION MUST DO DIFFERENTLY")
print("  1. -10 C.  Check the setpoint against the protocol before the first frame, not after.")
print("  2. 20 bias at each gain, at both ends.  Under a minute, and it removes the largest")
print("     single uncertainty in F_sky.")
print("  3. Export the guiding log.")
print("  4. Plan the meridian flip: either avoid it inside the window, or define both ROIs")
print("     twice and treat the halves as separate blocks.")
print("  5. Frame so a corner sits on genuinely dark sky.")
print()
print("None of the five needs new equipment and all five are free.  The expensive thing this")
print("session revealed is not on the list: it is that the rig spends "
      f"{100 * T_DEAD / (30 + T_DEAD):.0f}% of a 30 s")
print("sub-exposure night moving the mount, and that is a setting, not a fact.")